# Baseline｜完整因子池冻结、Top100 策略输入与 OOS Gate

本 Notebook 是薄的正式执行入口：D1 始终冻结完整 Stage 6 Provisional Factor Pool；策略阶段只使用 frozen ordering 的固定前 100 个因子。三种策略共享同一 Top100。默认在 OOS gate 前停止。

本流程只打印 Strategy Matrix 的基本 sanity 信息，不开发 coverage reporting；不重新筛选 Top100、不修改 Stage 6，也不提供 LightGBM 调参接口。

## Cell 1｜配置与正式 Stage 6 authority

绑定本次唯一正式 Stage 6 run、人工审阅过的 selection fingerprint，以及固定的 `TOP_K_STRATEGY_INPUT = 100`。

In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from factor_gfn.backtest import (
    BaselineFactorPoolFreezeInputs,
    LIGHTGBM_EARLY_STOPPING_PATIENCE,
    LIGHTGBM_FIXED_PARAMS,
    build_development_factor_matrices,
    build_stage5_data_context,
    build_stage6_evaluation_context,
    build_static_strategy_bundle,
    build_test_factor_matrix,
    evaluate_oos_baselines,
    freeze_baseline_factor_pool,
    freeze_development_factor_matrices,
    freeze_oos_baseline_evaluation,
    freeze_static_strategy_bundle,
    freeze_test_score_artifact,
    freeze_top100_strategy_input,
    generate_test_strategy_scores,
    load_verified_baseline_factor_pool,
    load_verified_development_factor_matrices,
    load_verified_strategy_bundle,
    load_verified_strategy_input,
    load_verified_test_labels,
    load_verified_test_score_artifact,
    unlock_verified_test_features,
)
from factor_gfn.backtest.baseline_factor_pool import OOS_UNTOUCHED
from factor_gfn.gfn import RealRewardDataPaths

HYBRID_RUN_ID = 'hybrid_5_15_k16_seed42_20260816T025559Z'
REVIEWED_SELECTION_FINGERPRINT = (
    'b96a163e797ec7d803fc423f6625f8614bd55ddfdf827a4bd11a157bd80484fb'
)
TOP_K_STRATEGY_INPUT = 100
CONFIRM_OOS_UNLOCK = False
RUNS_ROOT = REPO_ROOT / 'runs'
STAGE6_ROOT = RUNS_ROOT / 'stage6' / 'hybrid_provisional' / HYBRID_RUN_ID
DATA_PATHS = RealRewardDataPaths()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def unique_manifest(relative_pattern, label):
    matches = sorted(STAGE6_ROOT.glob(relative_pattern))
    if len(matches) != 1:
        raise RuntimeError(f'{label} 应恰好存在 1 个，实际为 {len(matches)} 个：{matches}')
    return matches[0].resolve()

TRAIN_REUSE_MANIFEST = unique_manifest('train_reuse/*/train_reuse_manifest.json', 'Train-reuse manifest')
train_reuse = read_json(TRAIN_REUSE_MANIFEST)
SOURCE_SET_MANIFEST = (STAGE6_ROOT / 'source_snapshots' / 'source_sets' / train_reuse['source_set_fingerprint'] / 'source_set_manifest.json').resolve()
CANDIDATE_IMPORT_MANIFEST = (STAGE6_ROOT / 'candidate_import' / train_reuse['candidate_registry_fingerprint'] / 'candidate_import_manifest.json').resolve()
COMPATIBILITY_MANIFEST = (STAGE6_ROOT / 'compatibility' / train_reuse['compatibility_audit_fingerprint'] / 'expression_compatibility_manifest.json').resolve()
TRAIN_ENTRY_MANIFEST = (STAGE6_ROOT / 'train_preparation' / 'train_preparation_entry_manifest.json').resolve()
TRAIN_PASS_MANIFEST = (STAGE6_ROOT / 'train_preparation' / 'train_pass_manifest' / 'train_pass_manifest.json').resolve()
VALIDATION_ENTRY_MANIFEST = (STAGE6_ROOT / 'validation_evaluation' / 'validation_evaluation_entry_manifest.json').resolve()
SELECTION_MANIFEST = unique_manifest('provisional_selection/*/enriched_selection_manifest.json', 'Enriched selection manifest')

manifest_paths = {
    'source_set': SOURCE_SET_MANIFEST,
    'candidate_import': CANDIDATE_IMPORT_MANIFEST,
    'compatibility': COMPATIBILITY_MANIFEST,
    'train_reuse': TRAIN_REUSE_MANIFEST,
    'train_entry': TRAIN_ENTRY_MANIFEST,
    'train_pass': TRAIN_PASS_MANIFEST,
    'validation_entry': VALIDATION_ENTRY_MANIFEST,
    'selection': SELECTION_MANIFEST,
}
missing = [str(path) for path in manifest_paths.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(f'正式 authority 输入缺失：{missing}')
selection = read_json(SELECTION_MANIFEST)
if selection.get('enriched_selection_fingerprint') != REVIEWED_SELECTION_FINGERPRINT:
    raise RuntimeError('Stage 6 selection fingerprint 与已审阅 fingerprint 不一致')
if TOP_K_STRATEGY_INPUT != 100:
    raise RuntimeError('当前正式 StrategyInput 合同只支持固定 Top100')

print('Frozen Baseline Factor Pool = full Stage 6 pool')
print('Strategy Input Cap = Top100 from frozen ordering')
print('Formal Hybrid Stage 6 run:', HYBRID_RUN_ID)
print('Reviewed selection fingerprint:', REVIEWED_SELECTION_FINGERPRINT)
print('Stage 6 retained count:', selection.get('counts', {}).get('retained'))
print('CONFIRM_OOS_UNLOCK:', CONFIRM_OOS_UNLOCK)

## Cell 2｜D1：冻结完整 Baseline Factor Pool

原样冻结 Stage 6 正式 selection 的全部成员、顺序和 Train direction；不重新筛选或去相关。

In [ ]:
freeze_inputs = BaselineFactorPoolFreezeInputs(
    source_set_manifest_path=SOURCE_SET_MANIFEST,
    candidate_import_manifest_path=CANDIDATE_IMPORT_MANIFEST,
    compatibility_manifest_path=COMPATIBILITY_MANIFEST,
    train_reuse_manifest_path=TRAIN_REUSE_MANIFEST,
    train_entry_manifest_path=TRAIN_ENTRY_MANIFEST,
    train_pass_manifest_path=TRAIN_PASS_MANIFEST,
    validation_entry_manifest_path=VALIDATION_ENTRY_MANIFEST,
    enriched_selection_manifest_path=SELECTION_MANIFEST,
)
pool_artifact = freeze_baseline_factor_pool(
    freeze_inputs,
    RUNS_ROOT,
    confirmed_for_freeze=True,
    reviewed_selection_fingerprint=REVIEWED_SELECTION_FINGERPRINT,
)
frozen_pool = load_verified_baseline_factor_pool(pool_artifact.manifest_path)
selection_retained = int(selection.get('counts', {}).get('retained', -1))
assert len(frozen_pool.records) == selection_retained
assert tuple(record.structural_hash for record in frozen_pool.records) == frozen_pool.ordered_structural_hashes
assert tuple(record.train_direction for record in frozen_pool.records) == frozen_pool.frozen_train_directions
assert frozen_pool.oos_status == OOS_UNTOUCHED
print('Factor Pool manifest:', frozen_pool.manifest_path)
print('Factor Pool fingerprint:', frozen_pool.baseline_factor_pool_fingerprint)
print('Frozen factor count:', len(frozen_pool.records))
print('Selection fingerprint:', frozen_pool.manifest['authorization']['reviewed_selection_fingerprint'])
print('OOS status:', frozen_pool.oos_status)

## Cell 3｜冻结固定 Top100 StrategyInput

只取完整 frozen ordering 的前 100 条，不排序、不重算 ranking、不看 coverage 或 OOS。该 artifact 同时绑定完整 pool fingerprint 和 Top100 前缀 fingerprint。

In [ ]:
strategy_input_artifact = freeze_top100_strategy_input(frozen_pool, RUNS_ROOT)
strategy_input = load_verified_strategy_input(strategy_input_artifact.manifest_path)
assert strategy_input.top_k == TOP_K_STRATEGY_INPUT
assert strategy_input.ordered_structural_hashes == frozen_pool.ordered_structural_hashes[:TOP_K_STRATEGY_INPUT]
assert strategy_input.frozen_train_directions == frozen_pool.frozen_train_directions[:TOP_K_STRATEGY_INPUT]
print('Frozen Factor Pool size:', len(frozen_pool.records))
print('Strategy Input Cap:', TOP_K_STRATEGY_INPUT)
print('Actual Strategy Input size:', len(strategy_input.records))
print('Strategy Input manifest:', strategy_input.manifest_path)
print('Strategy Input fingerprint:', strategy_input.strategy_input_fingerprint)

## Cell 4｜构建并冻结 Top100 Development Matrix

只读取 Train + Validation，并显式传入已验证的 StrategyInput。cleaning 后的 factor-specific non-finite 仅在 base-eligible 股票内填 0。

In [ ]:
development_context = build_stage6_evaluation_context(paths=DATA_PATHS)
development_matrices = build_development_factor_matrices(
    frozen_pool,
    development_context,
    strategy_input=strategy_input,
)
matrix_artifact = freeze_development_factor_matrices(development_matrices, RUNS_ROOT)
development_matrices = load_verified_development_factor_matrices(matrix_artifact.manifest_path)
assert len(development_matrices.feature_mapping) == TOP_K_STRATEGY_INPUT
assert development_matrices.strategy_input_fingerprint == strategy_input.strategy_input_fingerprint
train_rows = development_matrices.splits['train'].features.values.shape[0]
validation_rows = development_matrices.splits['validation'].features.values.shape[0]
remaining_nonfinite = sum(
    int((~np.isfinite(split.features.values)).sum())
    for split in development_matrices.splits.values()
)
print('Development Matrix manifest:', development_matrices.artifact_manifest_path)
print('Top100 factor count:', len(development_matrices.feature_mapping))
print('Train rows:', train_rows)
print('Validation rows:', validation_rows)
print('Remaining non-finite feature count:', remaining_nonfinite)
print('Base-eligible rows retained:', train_rows + validation_rows)

## Cell 5｜构建三种静态策略

Equal Weight、Fixed ICIR 和 LightGBM 共享同一个 Top100 矩阵。LightGBM 直接使用代码中冻结的 Baseline 参数和既有 early-stopping / final-refit 流程。

In [ ]:
built_bundle = build_static_strategy_bundle(frozen_pool, development_matrices)
assert built_bundle.strategy_input_fingerprint == strategy_input.strategy_input_fingerprint
assert len(built_bundle.ordered_structural_hashes) == TOP_K_STRATEGY_INPUT
icir_weights = np.asarray(built_bundle.strategies['fixed_icir'].weights, dtype=np.float64)
lightgbm_metadata = built_bundle.strategies['lightgbm'].metadata
print('Strategies:', tuple(built_bundle.strategies))
print('Shared Strategy Input count:', len(built_bundle.ordered_structural_hashes))
print('ICIR nonzero weights:', int(np.count_nonzero(icir_weights)))
print('ICIR weight min / median / max:', float(icir_weights.min()), float(np.median(icir_weights)), float(icir_weights.max()))
print('LightGBM best iteration:', lightgbm_metadata['best_iteration'])
print('LightGBM fixed parameters:', dict(LIGHTGBM_FIXED_PARAMS))
print('LightGBM early stopping patience:', LIGHTGBM_EARLY_STOPPING_PATIENCE)

## Cell 6｜冻结 Static Strategy Bundle

Bundle authority 同时绑定完整 Factor Pool fingerprint、StrategyInput fingerprint 和 Development Matrix fingerprint。

In [ ]:
bundle_artifact = freeze_static_strategy_bundle(built_bundle, RUNS_ROOT)
frozen_bundle = load_verified_strategy_bundle(bundle_artifact.manifest_path)
assert frozen_bundle.factor_pool_fingerprint == frozen_pool.baseline_factor_pool_fingerprint
assert frozen_bundle.strategy_input_fingerprint == strategy_input.strategy_input_fingerprint
assert frozen_bundle.ordered_structural_hashes == strategy_input.ordered_structural_hashes
print('Strategy Bundle manifest:', frozen_bundle.manifest_path)
print('Strategy Bundle fingerprint:', frozen_bundle.bundle_fingerprint)
print('Component paths:')
for relative_path in frozen_bundle.manifest['artifacts']:
    print(' -', frozen_bundle.manifest_path.parent / relative_path)

## Cell 7｜显式 OOS unlock gate（默认停止点）

运行到这里仍未读取 Test features 或 Test labels。只有回到 Cell 1，把 `CONFIRM_OOS_UNLOCK` 人工改为 `True` 并重新从头运行后，才执行后续两个 Cell。

In [ ]:
print('Factor Pool fingerprint:', frozen_pool.baseline_factor_pool_fingerprint)
print('Frozen factor count:', len(frozen_pool.records))
print('Strategy Input fingerprint:', strategy_input.strategy_input_fingerprint)
print('Strategy Input size:', len(strategy_input.records))
print('Strategy Bundle fingerprint:', frozen_bundle.bundle_fingerprint)
print('LightGBM best iteration:', frozen_bundle.strategies['lightgbm'].metadata['best_iteration'])
print('OOS status:', frozen_bundle.oos_status)
print('CONFIRM_OOS_UNLOCK:', CONFIRM_OOS_UNLOCK)
if not CONFIRM_OOS_UNLOCK:
    print('STOP：OOS gate 未打开。不要运行 Cell 8/9。')
else:
    print('OOS gate 已人工打开；可继续运行 Cell 8，然后运行 Cell 9。')

## Cell 8｜解锁后：Test features → Test scores → 先冻结 Score Artifact

此 Cell 的第一条有效语句就是 gate。它只构建 Top100 Test Matrix 和三策略 scores；在 score artifact 完成并重新验证之前不读取 Test labels。

In [ ]:
if not CONFIRM_OOS_UNLOCK:
    raise PermissionError('OOS gate closed: set CONFIRM_OOS_UNLOCK=True in Cell 1 and rerun from the top')

oos_context = build_stage5_data_context(paths=DATA_PATHS)
test_feature_context = unlock_verified_test_features(oos_context, frozen_pool, frozen_bundle)
test_matrix = build_test_factor_matrix(test_feature_context, frozen_pool, frozen_bundle)
assert test_matrix.features.factor_count == TOP_K_STRATEGY_INPUT
test_scores = generate_test_strategy_scores(frozen_bundle, test_matrix)
score_artifact = freeze_test_score_artifact(
    frozen_pool, frozen_bundle, test_matrix, test_scores, RUNS_ROOT
)
verified_scores = load_verified_test_score_artifact(
    score_artifact.manifest_path, frozen_pool, frozen_bundle, test_matrix
)
print('Test factor count:', test_matrix.features.factor_count)
print('Test rows:', test_matrix.features.values.shape[0])
print('Test Factor Matrix fingerprint:', test_matrix.fingerprint)
print('Frozen Test Score manifest:', verified_scores.manifest_path)
print('Frozen Test Score fingerprint:', verified_scores.fingerprint)
print('Test labels have not been loaded yet.')

## Cell 9｜解锁后：首次读取 Test labels，执行并冻结 OOS Evaluation

只有已冻结且重新验证的 Test Score Artifact 才能打开 label gateway。

In [ ]:
if not CONFIRM_OOS_UNLOCK:
    raise PermissionError('OOS gate closed')

test_labels = load_verified_test_labels(
    oos_context, frozen_pool, frozen_bundle, test_matrix, verified_scores
)
oos_evaluation = evaluate_oos_baselines(
    frozen_pool, frozen_bundle, test_matrix, verified_scores, test_labels
)
oos_artifact = freeze_oos_baseline_evaluation(oos_evaluation, RUNS_ROOT)
print('OOS evaluation manifest:', oos_artifact.manifest_path)
print('OOS evaluation fingerprint:', oos_artifact.fingerprint)
print('Reporting notebook:', REPO_ROOT / 'notebooks' / 'oos_baseline_evaluation.ipynb')
print('请把以上真实 OOS evaluation manifest 路径填入 reporting notebook。')